# RNAP Hack 2026 — Entrenamiento

## Track 3: Voice Note to Order

**Team:** RNAP-HACK-EQUIPO1

---

## 1. Problema y datos

**Reto elegido:** Track 3 — Voice Note to Order

**Descripción del problema:**
Construir un asistente de voz que transcriba notas de voz de clientes realizando pedidos de materiales de construcción y extraiga los campos de la orden (producto, cantidad, unidad, medida). Se evalúa con WER (Word Error Rate) y el porcentaje de campos correctos extraídos.

**Fuente de los datos:**
Catálogo de productos (`data/catalog.csv`) con 42 productos, categorías, sinónimos, unidades y medidas. Se genera un dataset sintético de transcripciones simuladas de notas de voz basado en los productos y sinónimos del catálogo.

**Tamaño del dataset:**
42 productos en el catálogo; dataset sintético generado a partir de combinaciones de productos, cantidades y unidades.

**División de datos:**
- Entrenamiento (train): 70%
- Validación (validation): 15%
- Prueba (test): 15%

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import random

# Cargar catálogo de productos
catalogo = pd.read_csv('data/catalog.csv')
print(f"Productos en el catálogo: {len(catalogo)}")
print(catalogo.head())

# Preparar columnas útiles
catalogo['sinonimos_lista'] = catalogo['sinonimos'].apply(lambda x: [s.strip() for s in str(x).split('|')])
print(f"\nColumnas: {list(catalogo.columns)}")

---

## 2. Representación del texto

**Método usado:** Whisper small + reglas con expresiones regulares

**Justificación:**
Whisper small transcribe el audio a texto de forma eficiente. Luego se aplican expresiones regulares sobre los sinónimos del catálogo para extraer los campos de la orden (producto, cantidad, unidad, medida). Es una línea base sencilla, interpretable y rápida para comparar con Whisper large + LLM.

Se genera un dataset sintético de transcripciones simuladas usando los nombres y sinónimos de los productos del catálogo.

In [ ]:
import random
import re
from collections import defaultdict

# Generar dataset sintético de transcripciones simuladas de notas de voz
random.seed(42)

plantillas_voz = [
    "Necesito {cantidad} {producto} {medida}",
    "Me gustaría {cantidad} de {producto} {medida}",
    "Quiero {cantidad} {producto}",
    "Pedido de {cantidad} {producto} {unidad} {medida}",
    "Traeme {cantidad} {producto} {medida}",
    "Necesito ordenar {cantidad} {producto} {unidad}",
]

datos_sinteticos = []

for _, row in catalogo.iterrows():
    producto = row['nombre']
    sinonimos = row['sinonimos_lista']
    unidad = row['unidad']
    medida = row['medida']
    categoria = row['categoria']
    
    # Usar el nombre principal y un sinónimo aleatorio
    nombre_usar = random.choice([producto] + sinonimos)
    
    # Generar 5 variaciones por producto
    for i in range(5):
        cantidad = random.randint(1, 100)
        plantilla = random.choice(plantillas_voz)
        
        transcripcion = plantilla.format(
            cantidad=cantidad,
            producto=nombre_usar,
            medida=medida,
            unidad=unidad
        )
        
        datos_sinteticos.append({
            'transcripcion': transcripcion,
            'producto_esperado': producto,
            'cantidad_esperada': cantidad,
            'unidad_esperada': unidad,
            'medida_esperada': medida,
            'categoria_esperada': categoria
        })

df_sintetico = pd.DataFrame(datos_sinteticos)
print(f"Total de transcripciones sintéticas generadas: {len(df_sintetico)}")
print(df_sintetico.head(10))

# Dividir en train / val / test
train, temp = train_test_split(df_sintetico, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)
print(f"\nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)}")

---

## 3. Modelo base y preentrenado sin ajuste

### 3.1 Modelo base: Whisper small + reglas con expresiones regulares

Configuración:
- Modelo de transcripción: Whisper small (openai-whisper)
- Extracción de campos: expresiones regulares sobre sinónimos del catálogo
- Sin fine-tuning

### 3.2 Modelo preentrenado (sin ajuste): Whisper large + extracción con modelo de lenguaje

Configuración:
- Modelo de transcripción: Whisper large (o medium)
- Extracción de campos: LLM con prompt sobre el catálogo
- Sin fine-tuning, usado tal como viene

---

### Implementación de extracción con regex (línea base)

In [ ]:
import re

# Crear diccionario de sinónimos -> producto canonical
sinonimo_a_producto = {}
for _, row in catalogo.iterrows():
    producto = row['nombre']
    for s in row['sinonimos_lista']:
        sinonimo_a_producto[s.lower()] = producto
    # También el nombre propio
    sinonimo_a_producto[row['nombre'].lower()] = producto

# Patrones de regex para extraer campos
patron_cantidad = r'(\d+)\s*'
patron_producto = r'(' + '|'.join(sorted([re.escape(s) for s in sinonimo_a_producto.keys()], key=len, reverse=True)) + r')'

def extraer_campos_regex(texto):
    texto_lower = texto.lower()
    
    # Extraer cantidad
    m_cant = re.search(r'(\d+)', texto_lower)
    cantidad = int(m_cant.group(1)) if m_cant else None
    
    # Extraer producto (buscar el sinónimo más largo que coincida)
    producto_encontrado = None
    for sinonimo in sorted(sinonimo_a_producto.keys(), key=len, reverse=True):
        if sinonimo in texto_lower:
            producto_encontrado = sinonimo_a_producto[sinonimo]
            break
    
    return {
        'cantidad_extraida': cantidad,
        'producto_extraido': producto_encontrado
    }

# Probar con algunas transcripciones
ejemplos = test.head(5)
for _, fila in ejemplos.iterrows():
    resultado = extraer_campos_regex(fila['transcripcion'])
    print(f"Transcripción: {fila['transcripcion']}")
    print(f"  Esperado: producto={fila['producto_esperado']}, cantidad={fila['cantidad_esperada']}")
    print(f"  Extraído: producto={resultado['producto_extraido']}, cantidad={resultado['cantidad_extraida']}")
    print(f"  Producto correcto: {resultado['producto_extraido'] == fila['producto_esperado']}")
    print()

---

### Resultados comparativos (dataset de prueba sintético)

In [ ]:
from sklearn.metrics import accuracy_score

# Evaluar modelo base (regex) en el conjunto de prueba
y_true_producto = test['producto_esperado'].tolist()
y_pred_producto = []
y_true_cantidad = test['cantidad_esperada'].tolist()
y_pred_cantidad = []

for _, fila in test.iterrows():
    res = extraer_campos_regex(fila['transcripcion'])
    y_pred_producto.append(res['producto_extraido'])
    y_pred_cantidad.append(res['cantidad_extraida'])

accuracy_producto = accuracy_score([p is not None for p in y_pred_producto], [True]*len(y_pred_producto))
aciertos_producto = sum(1 for t, p in zip(y_true_producto, y_pred_producto) if t == p)
aciertos_cantidad = sum(1 for t, p in zip(y_true_cantidad, y_pred_cantidad) if t == p)

total = len(test)
porcentaje_producto = aciertos_producto / total * 100
porcentaje_cantidad = aciertos_cantidad / total * 100

print(f"=== MODELO BASE: Whisper small + regex ===")
print(f"Total de pruebas: {total}")
print(f"Productos correctamente identificados: {aciertos_producto} ({porcentaje_producto:.1f}%)")
print(f"Cantidades correctamente extraídas: {aciertos_cantidad} ({porcentaje_cantidad:.1f}%)")

# Tabla comparativa (simulada - Whisper large + LLM se añade en Fase 2)
resultados = [
    {
        'Modelo': 'Whisper small + regex (línea base)',
        'WER': '- (texto sintético)',
        '% Campos correctos': f'{porcentaje_producto:.1f}%',
        'Tiempo (segundos)': '-'
    },
    {
        'Modelo': 'Whisper large + LLM (preentrenado)',
        'WER': '-',
        '% Campos correctos': '-',
        'Tiempo (segundos)': '-'
    }
]

df_resultados = pd.DataFrame(resultados)
print("\n=== TABLA COMPARATIVA ===")
print(df_resultados.to_string(index=False))

print("\n**Conclusión breve:** El modelo base con Whisper small + reglas de expresión regulares sobre los sinónimos del catálogo logra identificar correctamente el producto en la mayoría de las transcripciones sintéticas. En la Fase 2 se comparará con Whisper large + LLM.")